# Data cleaning: death points and short coins

Reads every coin from the **raw data folder** (`RAW_DIR`: `{base}_metadata.json` + `{base}_ohlcv.csv` pairs written by the collection pipeline), cleans it, and writes the result to the **processed data folder** (`PROCESSED_DIR`). The raw files are **never modified**; rerunning rebuilds the processed folder from scratch.

**1. Gap filling.** The series is put on a continuous 1-minute grid, and **every zero-volume minute** becomes a **flat candle at the last real close**: open = high = low = close, volume 0. This covers minutes missing from the file, the API's empty intervals, and minutes filled the old way (previous candle copied). A `backfilled` flag (1 = zero-volume minute) is derived from the volume. This is the same as the pipeline and the live bot.

**2. Volume mark.** A **48-hour** window slides minute by minute. The first window whose total volume is below **1,000 SOL** marks the coin dead, and the data is cut at that window's **end**. The first 48 hours are a **launch grace period**: only windows that **start** at or after the 48-hour mark count. This is the same rule as the collection pipeline, which stops collecting there; the data the pipeline keeps is therefore never cut differently here.

**3. Label (pump-rate) mark.** Per minute, the label as in the dataset notebook: 1 if the close rises ≥ 10% within the next 30 minutes. A **7-day** window slides over the labels; once a window's average falls below **1%**, the data is cut at that window's **beginning**. This mark is determined in hindsight: it ends a coin's data where its pump activity has already stopped for good.

**4. Cut.** At whichever mark keeps **fewer** rows.

**5. Short coins.** Coins with less than **12 hours** of data after cutting are left out of the processed folder. Trading starts 12 hours after a coin's first minute, so these coins could never be traded anyway.

**Output** (`PROCESSED_DIR`):
- `{base}_ohlcv_cleaned.csv`: the cleaned 1-minute series;
- `{base}_metadata.json`: the raw metadata plus a `death_point` block recording both marks and how much each removed.

The run reports per coin and for the whole dataset how much each mark removes, and which coins were left out.

CSVs in the raw folder **without a metadata file** are reported but not processed. Run the one-time notebook `fix_orphan_metadata.ipynb` first to give them one.

In [ ]:
#
# ============================================================
# CONFIGURATION
# ============================================================
#

# ── REPOSITORY FOLDERS ───────────────────────────────────────────────────────
# All paths are relative to the repository's top folder, found from where the notebook runs:
#   data/                         raw_data/, processed_data/, dataset_v9/        (not committed)
#   research_development/<name>/  each notebook's outputs                        (not committed)
#   live_pipeline/artifacts/      the chosen models and strategy files, copied there by hand
from pathlib import Path


def _repo_root():
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / 'live_pipeline').is_dir() and (p / 'development_notebooks').is_dir():
            return p
    raise RuntimeError("run this notebook inside the repository: no folder above it holds "
                       "live_pipeline/ and development_notebooks/")


REPO = _repo_root()
REPO_DATA = REPO / 'data'
REPO_RESULTS = REPO / 'research_development'
REPO_ARTIFACTS = REPO / 'live_pipeline' / 'artifacts'

RAW_DIR       = str(REPO_DATA / "raw_data")         # input: {base}_metadata.json + {base}_ohlcv.csv pairs (never modified)
PROCESSED_DIR = str(REPO_DATA / "processed_data")   # output: {base}_ohlcv_cleaned.csv + {base}_metadata.json per kept coin
TIMEZONE      = "Europe/Amsterdam"
CLEAR_PROCESSED = True               # True: empty PROCESSED_DIR of earlier cleaning output first, so it
                                     # always matches the current raw data and settings exactly

# Volume mark — the same rule as the collection pipeline
WINDOW_HOURS         = 48            # sliding window length
VOLUME_THRESHOLD_SOL = 1000          # a 48h window with less total volume than this = dead
GRACE_HOURS          = 48            # launch grace: only windows STARTING at or after this mark count

# Label (pump-rate) mark — the label matches the dataset-creation notebook
LABEL_FORWARD_MINUTES = 30           # forward window for the label
LABEL_THRESHOLD_PCT   = 0.10         # label = 1 if close rises >= this within the forward window
LABEL_WINDOW_DAYS     = 7            # moving window for the average label
LABEL_MIN_AVG         = 0.01         # a 7d window with an average label below this (1%) = pump-dead;
                                     # the data is cut at the window's BEGINNING

MIN_DATA_HOURS = 12                  # coins with less data than this after cutting are left out
                                     # (trading starts 12h after a coin's first minute)

In [ ]:
import os
import json
import numpy as np
import pandas as pd


# ============================================================
# BACKFILL  (continuous 1-minute grid)  — same as the pipeline and the live bot
# ============================================================

def backfill_missing_candles(df, verbose=False):
    """Put the series on a continuous 1-minute grid and make EVERY zero-volume minute a FLAT
    candle at the last real close: open = high = low = close, volume = 0 (the same as the
    pipeline and the live bot). This covers minutes missing from the file, the API's empty
    intervals, and minutes filled the old way (previous candle copied). Adds 'backfilled'
    (1 = zero-volume minute, 0 = real candle), derived from the volume."""
    if df is None or len(df) == 0:
        return df

    df = df.drop(columns=["backfilled"], errors="ignore")
    df = df.drop_duplicates(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)

    full_range = pd.date_range(start=df["timestamp"].min(), end=df["timestamp"].max(), freq="1min")
    if verbose:
        print(f"  Missing minutes to fill: {len(full_range) - len(df)}")

    df_filled = df.set_index("timestamp").reindex(full_range)
    df_filled["volume"] = df_filled["volume"].fillna(0)
    zero = (df_filled["volume"] <= 0).to_numpy()
    last_close = df_filled["close"].where(~zero).ffill()          # last REAL close
    last_close = last_close.fillna(df_filled["close"].bfill())    # zero-volume minutes before any real candle
    for col in ("open", "high", "low", "close"):
        df_filled.loc[zero, col] = last_close[zero]
    df_filled["backfilled"] = zero.astype(int)
    df_filled = df_filled.reset_index().rename(columns={"index": "timestamp"})
    return df_filled


# ============================================================
# VOLUME MARK  — the same moving-window rule as the collection pipeline
# ============================================================

def find_death_point(df, window_hours=48, volume_threshold=1000, grace_hours=48):
    """
    Slide a `window_hours` window minute by minute and find the FIRST window whose total
    volume is below `volume_threshold` (SOL). The first `grace_hours` are a launch grace
    period: only windows that START at or after that mark count. The data is cut at that
    window's END. Assumes a continuous 1-minute grid (call after backfill_missing_candles).
    Returns (kept_rows, death_ts): kept_rows = leading rows to KEEP (through the window's
    last row); death_ts = last kept timestamp. No death -> (len(df), None).
    """
    if df is None or len(df) == 0:
        return 0, None
    df = df.sort_values("timestamp").reset_index(drop=True)
    n = len(df)
    wm = int(window_hours * 60)
    gm = int(grace_hours * 60)
    if n < wm:
        return n, None
    vol = df["volume"].to_numpy(dtype=float)
    csum = np.concatenate(([0.0], np.cumsum(vol)))
    win_sums = csum[wm:] - csum[:-wm]                  # win_sums[k] = volume of rows [k, k+wm-1]
    below = np.where(win_sums < volume_threshold)[0]
    below = below[below >= gm]                         # windows starting in the grace period don't count
    if len(below) == 0:
        return n, None
    end_i = int(below[0]) + wm - 1                     # last row of the first dead window
    return end_i + 1, df.loc[end_i, "timestamp"]       # keep rows [0, end_i]


# ============================================================
# LABEL-BASED (PUMP-RATE) DEATH DETECTION
# ============================================================
# The label matches the dataset-creation notebook exactly: for row i,
#   label = 1  iff  max(close[i+1 : i+1+FORWARD]) >= close[i] * (1 + THRESHOLD)
# (close-based, forward-looking; the last FORWARD rows are 0 = unconfirmable).
# A coin is "pump-dead" once a moving window of LABEL_WINDOW_DAYS has an average
# label below LABEL_MIN_AVG; the data is ended at that window's end.

def compute_labels(df, forward_minutes=30, threshold_pct=0.10):
    """Per-row 0/1 label, identical definition to the dataset notebook."""
    close = df["close"].to_numpy(dtype=float)
    n = len(close)
    labels = np.zeros(n, dtype=float)
    if n > forward_minutes:
        roll = pd.Series(close).rolling(window=forward_minutes, min_periods=forward_minutes).max().to_numpy()
        fm = np.full(n, np.nan)
        fm[:n - forward_minutes] = roll[forward_minutes:]        # fm[i] = max(close[i+1 : i+1+forward])
        with np.errstate(invalid="ignore", divide="ignore"):
            pct = (fm - close) / close
        labels = np.where(np.isfinite(pct) & (pct >= threshold_pct), 1.0, 0.0)
    return labels


def find_label_death_point(df, labels, window_days=7, min_avg=0.01):
    """
    Pump-rate death: the FIRST `window_days` window whose average label is below `min_avg`.
    The data is cut at the window's BEGINNING (keep everything strictly before the window),
    which also eliminates coins that are exactly one window long. Returns (kept_rows, death_ts):
    kept_rows = leading rows to KEEP (= the window's start index); death_ts = last kept
    timestamp (None if the very first window is already dead -> keep nothing).
    No death -> (len(df), None).
    """
    n = len(labels)
    wm = int(window_days * 1440)
    if n < wm:
        return n, None                                 # not enough data for a full window
    csum = np.concatenate(([0.0], np.cumsum(labels)))
    win_means = (csum[wm:] - csum[:-wm]) / wm          # mean label over [k, k+wm-1]
    below = np.where(win_means < min_avg)[0]
    if len(below) == 0:
        return n, None
    k = int(below[0])                                  # window START -> cut here
    death_ts = df.loc[k - 1, "timestamp"] if k >= 1 else None
    return k, death_ts                                 # keep rows [0, k-1]



# ============================================================
# COMBINED ANALYSIS  —  both marks, crop at the earlier one
# ============================================================

def analyze_and_crop(df):
    """`df` is a backfilled 1-minute frame. Apply the VOLUME mark (48h window after a 48h grace,
    cut at its end) and the LABEL mark (7d window, cut at its beginning), report how much each
    removes, and cut at whichever keeps FEWER rows. Returns a dict with the cut frame + all stats."""
    df = df.sort_values("timestamp").reset_index(drop=True)
    n = len(df)

    kept_vol, vol_ts = find_death_point(df, WINDOW_HOURS, VOLUME_THRESHOLD_SOL, GRACE_HOURS)
    labels = compute_labels(df, LABEL_FORWARD_MINUTES, LABEL_THRESHOLD_PCT)
    kept_lbl, lbl_ts = find_label_death_point(df, labels, LABEL_WINDOW_DAYS, LABEL_MIN_AVG)

    vol_removed = n - kept_vol
    lbl_removed = n - kept_lbl
    combined_kept = min(kept_vol, kept_lbl)
    df_cropped = df.iloc[:combined_kept].copy()

    if combined_kept >= n:
        which = None
    elif kept_vol <= kept_lbl:
        which = "volume"
    else:
        which = "label"

    combined_ts = df.loc[combined_kept - 1, "timestamp"] if combined_kept > 0 else None
    return {"rows_full": n, "df_cropped": df_cropped, "rows_kept": combined_kept,
            "vol_death_ts": vol_ts, "lbl_death_ts": lbl_ts, "combined_death_ts": combined_ts,
            "vol_removed": vol_removed, "lbl_removed": lbl_removed,
            "combined_removed": n - combined_kept, "which_first": which,
            "is_dead": combined_kept < n}



def _death_point_block(a):
    """metadata death_point block recording BOTH marks + how much each removed."""
    def iso(t):
        return t.isoformat() if t is not None else None
    return {
        "is_dead": bool(a["is_dead"]),
        "death_timestamp": iso(a["combined_death_ts"]),
        "cropped_by": a["which_first"],
        "rows_full": a["rows_full"],
        "rows_removed_combined": a["combined_removed"],
        "volume_mark": {
            "death_timestamp": iso(a["vol_death_ts"]),
            "window_hours": WINDOW_HOURS,
            "volume_threshold_sol": VOLUME_THRESHOLD_SOL,
            "grace_hours": GRACE_HOURS,
            "rows_removed": a["vol_removed"],
        },
        "label_mark": {
            "death_timestamp": iso(a["lbl_death_ts"]),
            "window_days": LABEL_WINDOW_DAYS,
            "min_avg_label": LABEL_MIN_AVG,
            "forward_minutes": LABEL_FORWARD_MINUTES,
            "threshold_pct": LABEL_THRESHOLD_PCT,
            "rows_removed": a["lbl_removed"],
        },
    }


def _fmt_minutes(m):
    """Row count on a 1-min grid -> 'Xd Yh'."""
    m = int(m); d = m // 1440; h = (m % 1440) // 60
    return f"{d}d {h}h"


def summarize_removals(rows, title):
    """Dataset-level data-removed totals for both marks (over cut + left-out coins), plus a
    count of coins left out (< MIN_DATA_HOURS of data after cutting)."""
    used = [r for r in rows if r.get("status") in ("ok", "removed")]
    removed = [r for r in used if r.get("removed_full")]
    full = sum(r.get("rows_full", 0) for r in used)
    vol = sum(r.get("vol_removed", 0) for r in used)
    lbl = sum(r.get("lbl_removed", 0) for r in used)
    comb = sum(r.get("combined_removed", 0) for r in used)
    print(f"\n{title}  —  {len(used)} coins, {full} rows ({_fmt_minutes(full)}) total")
    if full > 0:
        print(f"  volume mark   would remove {vol:>10d} rows  ({vol/full*100:5.1f}%, {_fmt_minutes(vol)})")
        print(f"  label  mark   would remove {lbl:>10d} rows  ({lbl/full*100:5.1f}%, {_fmt_minutes(lbl)})")
        print(f"  combined crop  removed     {comb:>10d} rows  ({comb/full*100:5.1f}%, {_fmt_minutes(comb)})")
    print(f"  COINS LEFT OUT (< {MIN_DATA_HOURS}h after the cut): {len(removed)}")
    if removed:
        print(f"    {[r['base_name'] for r in removed]}")
    return {"rows_full": full, "vol_removed": vol, "lbl_removed": lbl,
            "combined_removed": comb, "fully_removed": len(removed)}



# ============================================================
# ATOMIC WRITES
# ============================================================

def _atomic_write_csv(df, path):
    tmp = f"{path}.tmp"
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def _atomic_write_json(obj, path):
    tmp = f"{path}.tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)


# ============================================================
# COIN LOADING
# ============================================================

def _load_csv(path, timezone):
    """Read an OHLCV CSV and normalise to tz-aware timestamps + float OHLCV."""
    df = pd.read_csv(path)
    if "backfilled" in df.columns:
        df = df.drop(columns=["backfilled"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True).dt.tz_convert(timezone)
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = df[col].astype(float)
    return df.drop_duplicates(subset=["timestamp"]).sort_values("timestamp").reset_index(drop=True)


def _resolve_csv(meta, base_name, data_folder):
    """Find the coin's CSV inside the raw data folder. Prefers the raw series, then a
    cleaned series (older archives)."""
    files = meta.get("data_files", {})
    candidates = [
        os.path.join(data_folder, f"{base_name}_ohlcv.csv"),
        os.path.join(data_folder, f"{base_name}_ohlcv_cleaned.csv"),
    ]
    for key in ("raw_data", "cleaned_data"):
        p = files.get(key)
        if p:
            candidates.append(os.path.join(data_folder, os.path.basename(p)))
    for c in candidates:
        if os.path.exists(c):
            return c
    return None


def list_metadata_files(data_folder):
    """Return [(base_name, meta_path)] for every metadata file in the data folder."""
    pairs = []
    if not os.path.exists(data_folder):
        print(f"Data folder not found: {data_folder}")
        return pairs
    for fname in sorted(os.listdir(data_folder)):
        if fname.endswith("_metadata.json"):
            pairs.append((fname[:-len("_metadata.json")], os.path.join(data_folder, fname)))
    return pairs


# ============================================================
# PROCESS ONE COIN  (reads RAW_DIR, writes PROCESSED_DIR; raw files are never touched)
# ============================================================

def process_coin(base_name, meta_path):
    """Backfill, apply both marks, cut at whichever keeps fewer rows. A coin left with less
    than MIN_DATA_HOURS of data is left out of PROCESSED_DIR; otherwise its cleaned CSV and its
    metadata (+ a death_point block) are written there."""
    min_rows = int(MIN_DATA_HOURS * 60)
    with open(meta_path, "r") as f:
        meta = json.load(f)
    symbol = meta.get("token_info", {}).get("token_symbol", base_name.split("_")[0].upper())

    csv_path = _resolve_csv(meta, base_name, RAW_DIR)
    if csv_path is None:
        return {"base_name": base_name, "symbol": symbol, "status": "no_csv",
                "rows_full": 0, "rows_out": 0, "vol_removed": 0, "lbl_removed": 0,
                "combined_removed": 0, "which_first": None, "is_dead": None, "removed_full": False,
                "vol_death_ts": None, "lbl_death_ts": None, "combined_death_ts": None}

    df = _load_csv(csv_path, TIMEZONE)
    df = backfill_missing_candles(df)
    a = analyze_and_crop(df)

    def iso(t):
        return t.isoformat() if t is not None else None

    if a["rows_kept"] < min_rows:
        # less than MIN_DATA_HOURS of usable data -> left out of the processed folder
        return {"base_name": base_name, "symbol": symbol, "status": "removed",
                "rows_full": a["rows_full"], "rows_out": 0,
                "vol_removed": a["vol_removed"], "lbl_removed": a["lbl_removed"],
                "combined_removed": a["rows_full"], "which_first": a["which_first"],
                "is_dead": True, "removed_full": True,
                "vol_death_ts": iso(a["vol_death_ts"]), "lbl_death_ts": iso(a["lbl_death_ts"]),
                "combined_death_ts": iso(a["combined_death_ts"]), "csv": os.path.basename(csv_path)}

    out_csv = os.path.join(PROCESSED_DIR, f"{base_name}_ohlcv_cleaned.csv")
    out_meta = os.path.join(PROCESSED_DIR, f"{base_name}_metadata.json")
    _atomic_write_csv(a["df_cropped"], out_csv)
    meta_out = dict(meta)
    meta_out["death_point"] = _death_point_block(a)
    # paths relative to the repository, so the metadata holds no machine-specific paths
    _rel = lambda p: Path(p).resolve().relative_to(REPO).as_posix()
    meta_out["data_files"] = dict(meta.get("data_files", {}), raw_data=_rel(csv_path), cleaned_data=_rel(out_csv))
    _atomic_write_json(meta_out, out_meta)

    return {"base_name": base_name, "symbol": symbol, "status": "ok",
            "rows_full": a["rows_full"], "rows_out": len(a["df_cropped"]),
            "vol_removed": a["vol_removed"], "lbl_removed": a["lbl_removed"],
            "combined_removed": a["combined_removed"], "which_first": a["which_first"],
            "is_dead": a["is_dead"], "removed_full": False,
            "vol_death_ts": iso(a["vol_death_ts"]), "lbl_death_ts": iso(a["lbl_death_ts"]),
            "combined_death_ts": iso(a["combined_death_ts"]), "csv": os.path.basename(csv_path)}

In [ ]:
# ============================================================
# RUN — clean every coin of RAW_DIR into PROCESSED_DIR
# ============================================================

os.makedirs(PROCESSED_DIR, exist_ok=True)
if CLEAR_PROCESSED:                                    # only this notebook's own output files
    old = [f for f in os.listdir(PROCESSED_DIR) if f.endswith(("_ohlcv_cleaned.csv", "_metadata.json"))]
    for f in old:
        os.remove(os.path.join(PROCESSED_DIR, f))
    print(f"Cleared {len(old)} earlier output file(s) from '{PROCESSED_DIR}'")

pairs = list_metadata_files(RAW_DIR)
total = len(pairs)
print(f"Found {total} metadata file(s) in '{RAW_DIR}'  ->  writing to '{PROCESSED_DIR}' (raw files untouched)\n")
print("=" * 80)

results = []
for i, (base_name, meta_path) in enumerate(pairs, 1):
    print(f"\n[{i}/{total}] {base_name}")
    try:
        res = process_coin(base_name, meta_path)
    except Exception as e:
        print(f"  ERROR: {e}")
        res = {"base_name": base_name, "symbol": None, "status": f"error: {e}",
               "rows_full": 0, "rows_out": 0, "vol_removed": 0, "lbl_removed": 0,
               "combined_removed": 0, "which_first": None, "is_dead": None, "removed_full": False}
    if res["status"] == "removed":
        print(f"  LEFT OUT (< {MIN_DATA_HOURS}h after the cut) — was {res['rows_full']} rows")
    elif res["status"] == "ok":
        if res["is_dead"]:
            print(f"  DEAD (by {res['which_first']}) — {res['rows_full']} -> {res['rows_out']} rows | "
                  f"volume removes {res['vol_removed']} ({_fmt_minutes(res['vol_removed'])}), "
                  f"label removes {res['lbl_removed']} ({_fmt_minutes(res['lbl_removed'])})")
        else:
            print(f"  alive — neither mark triggered ({res['rows_full']} rows)")
    elif res["status"] == "no_csv":
        print(f"  ! no matching CSV found in '{RAW_DIR}' — NOT analysed (name mismatch?)")
    results.append(res)

# ---------------- Coverage report ----------------
ok       = [r for r in results if r["status"] == "ok"]
removed  = [r for r in results if r["status"] == "removed"]
no_csv   = [r for r in results if r["status"] == "no_csv"]
errors   = [r for r in results if str(r["status"]).startswith("error")]
dead     = [r for r in ok if r["is_dead"]]
alive    = [r for r in ok if r["is_dead"] is False]

# metadata bases vs CSV bases -> catch names that didn't line up in either direction
meta_bases = {b for b, _ in pairs}
csv_bases = set()
if os.path.exists(RAW_DIR):
    for f in os.listdir(RAW_DIR):
        for suf in ("_ohlcv_cleaned.csv", "_ohlcv.csv"):
            if f.endswith(suf):
                csv_bases.add(f[:-len(suf)]); break
orphan_csvs = sorted(csv_bases - meta_bases)   # CSVs with no metadata partner

pct = ((len(ok) + len(removed)) / total * 100) if total else 0.0
print("\n" + "=" * 80)
print(f"Analysed {len(ok) + len(removed)}/{total} metadata files  ({(len(ok)+len(removed))/total*100 if total else 0:.1f}%)")
print(f"  dead (cut): {len(dead)}   alive: {len(alive)}   LEFT OUT (< {MIN_DATA_HOURS}h): {len(removed)}")
if removed:
    print(f"    left out: {[r['base_name'] for r in removed]}")
if no_csv:
    print(f"  NOT analysed — no matching CSV ({len(no_csv)}): {[r['base_name'] for r in no_csv]}")
if errors:
    print(f"  errors ({len(errors)}): {[(r['base_name'], r['status']) for r in errors]}")
if orphan_csvs:
    print(f"  CSVs with no metadata partner ({len(orphan_csvs)}), NOT processed — run fix_orphan_metadata.ipynb "
          f"once to give them a metadata file: {orphan_csvs}")
if pct < 100.0:
    print("  -> coverage < 100%: check the filenames above for pairs that didn't line up.")

# ---------------- Data-removed totals (metadata coins) ----------------
summarize_removals(results, "DATA REMOVED — whole dataset")
print(f"\n{len(ok)} cleaned coin(s) written to '{PROCESSED_DIR}'")